# 03 · PCA desde cero: varianza, autovectores y lo que se pierde

**Módulo 5 · Sesión 12** — Aprendizaje no supervisado

## Objetivos

El análisis de componentes principales (PCA) busca las direcciones de **mayor varianza**
de los datos y proyecta sobre ellas. Es la herramienta de reducción de dimensionalidad más
usada, y también la más malinterpretada: se usa como si "mayor varianza" significara
"más informativo", y no es lo mismo. Este notebook lo construye desde la matriz de
covarianza y la SVD del módulo 1, y mide qué se gana y qué se pierde:

1. Calcular los componentes principales **a mano** de dos formas —autovectores de la
   covarianza y SVD de los datos centrados— y verificar que coinciden con `PCA` de
   `scikit-learn`.
2. Ver por qué **estandarizar** cambia por completo el resultado sobre Wine Quality.
3. Leer las **cargas** y el **scree plot**, decidir cuántos componentes conservar, y
   conectar los dos primeros con los grupos del notebook 02.
4. Usar PCA como **compresión** sobre imágenes de dígitos, midiendo el error de
   reconstrucción.
5. Medir dos límites: cuando las variables están **incorreladas** PCA no comprime nada
   (`rendimiento-estudiantes.csv`, módulos 1 y 3); y cuando la información está en una
   dirección de **poca varianza**, PCA la descarta — con un ejemplo sintético y con la
   calidad del vino.

La teoría está en `03-reduccion-dimensionalidad.md`.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. La idea en dos dimensiones

Dos variables correlacionadas. La nube tiene una dirección "larga" y una "corta". PCA
encuentra esas direcciones: la primera componente es la dirección $\mathbf{v}_1$ (de
norma 1) que **maximiza la varianza** de la proyección $\mathbf{X}\mathbf{v}_1$; la
segunda, la que la maximiza entre las ortogonales a la primera; y así.

La solución es un resultado de álgebra lineal (módulo 1, S3): las direcciones son los
**autovectores** de la matriz de covarianza $\mathbf{S} = \mathbf{X}_c^\top \mathbf{X}_c /
(n-1)$, con $\mathbf{X}_c$ los datos centrados, y la varianza a lo largo de cada una es el
**autovalor** correspondiente.

In [ ]:
n = 300
X2 = rng.multivariate_normal(mean=[0, 0], cov=[[3.0, 2.2], [2.2, 2.0]], size=n)
X2_c = X2 - X2.mean(axis=0)

S = X2_c.T @ X2_c / (n - 1)                         # matriz de covarianza
autovalores, autovectores = np.linalg.eigh(S)       # eigh: simétrica → autovalores reales, ordenados ascendente
orden = np.argsort(autovalores)[::-1]
autovalores, autovectores = autovalores[orden], autovectores[:, orden]

print("Covarianza:\n", S.round(3))
print("Autovalores (varianza a lo largo de cada componente):", autovalores.round(3))
print("Fracción de varianza explicada:", (autovalores / autovalores.sum()).round(3))
print("Comprobación: la suma de autovalores es la varianza total:", np.isclose(autovalores.sum(), np.trace(S)))

fig, eje = plt.subplots(figsize=(6, 6))
eje.scatter(X2_c[:, 0], X2_c[:, 1], s=10, alpha=0.6)
for j, color in enumerate(["C3", "C2"]):
    v = autovectores[:, j] * 2 * np.sqrt(autovalores[j])
    eje.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=color, lw=3))
    eje.text(*(v * 1.15), f"PC{j + 1}", color=color, fontsize=13, fontweight="bold")
eje.set_aspect("equal")
eje.set_title("Los componentes principales: autovectores de la covarianza\n(flechas de longitud 2 desviaciones)")
plt.show()

### La misma respuesta por la SVD

El módulo 1 (notebook 03) terminó en la descomposición en valores singulares
$\mathbf{X}_c = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$. Como
$\mathbf{X}_c^\top\mathbf{X}_c = \mathbf{V}\boldsymbol{\Sigma}^2\mathbf{V}^\top$, las
columnas de $\mathbf{V}$ son los autovectores de la covarianza y
$\sigma_j^2 / (n-1)$ son los autovalores. Es lo que hace `scikit-learn` por dentro (es más
estable numéricamente que formar la covarianza), y las **puntuaciones** —las coordenadas
de cada punto sobre los componentes— son $\mathbf{X}_c\mathbf{V} = \mathbf{U}\boldsymbol{\Sigma}$.

In [ ]:
U, sigma, Vt = np.linalg.svd(X2_c, full_matrices=False)
pca2 = PCA().fit(X2)

print("Autovalores por SVD (σ²/(n-1)):", (sigma**2 / (n - 1)).round(3))
print("explained_variance_ de scikit-learn:", pca2.explained_variance_.round(3))
# Los autovectores están definidos salvo signo: comparamos en valor absoluto.
print("Componentes coinciden (salvo signo):", np.allclose(np.abs(Vt), np.abs(autovectores.T)),
      "| con scikit-learn:", np.allclose(np.abs(pca2.components_), np.abs(Vt)))
puntuaciones = X2_c @ Vt.T
print("Puntuaciones coinciden con transform (salvo signo):",
      np.allclose(np.abs(puntuaciones), np.abs(pca2.transform(X2))))

> **Los signos de los componentes son arbitrarios.** $\mathbf{v}$ y $-\mathbf{v}$ definen
> la misma dirección; distintas implementaciones (o distintas corridas) pueden devolver
> uno u otro. Nunca interpretes "PC1 positivo" sin mirar las cargas.

## 2. Wine Quality: por qué hay que estandarizar

PCA maximiza varianza, y la varianza depende de las unidades. Sobre las 11 variables de
Wine Quality **sin estandarizar**:

In [ ]:
vinos = pd.read_csv("../datos/wine-quality.csv").drop_duplicates().reset_index(drop=True)
X = vinos.drop(columns=["quality", "tipo"])
tipo = vinos["tipo"]
calidad = vinos["quality"]

pca_crudo = PCA().fit(X)
print("Varianza explicada sin estandarizar:", pca_crudo.explained_variance_ratio_.round(3))
print("PC1 sin estandarizar, cargas:")
print(pd.Series(pca_crudo.components_[0], index=X.columns).round(3).to_string())

El 95 % de la "varianza" está en un solo componente que es, casi exactamente,
`total_sulfur_dioxide` (carga 0.97): la variable con las unidades más grandes. PCA no
encontró estructura; encontró las unidades. Estandarizando, cada variable aporta
varianza 1 y la varianza total es 11:

In [ ]:
escalador = StandardScaler()
X_esc = escalador.fit_transform(X)
pca = PCA().fit(X_esc)

varianza = pd.DataFrame({
    "autovalor": pca.explained_variance_,
    "% varianza": 100 * pca.explained_variance_ratio_,
    "% acumulado": 100 * np.cumsum(pca.explained_variance_ratio_),
}, index=[f"PC{j + 1}" for j in range(X.shape[1])])
print(varianza.round(2).to_string())

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].bar(varianza.index, varianza["% varianza"])
ejes[0].axhline(100 / X.shape[1], color="gray", ls="--", label="1/p: lo que aporta una variable sola")
ejes[0].set_ylabel("% de la varianza total")
ejes[0].set_title("Scree plot")
ejes[0].legend()
ejes[1].plot(range(1, X.shape[1] + 1), varianza["% acumulado"], "o-")
ejes[1].axhline(80, color="gray", ls="--", label="80 %")
ejes[1].set_xlabel("Número de componentes")
ejes[1].set_ylabel("% acumulado")
ejes[1].set_title("Varianza acumulada")
ejes[1].legend()
plt.show()

### Cuántos componentes conservar

No hay una respuesta única; hay tres reglas habituales y dan cosas distintas:

- **Umbral de varianza acumulada** (80 % o 90 %): aquí 5 componentes rozan el 80 %
  (79.5) y 7 pasan el 90 %.
- **Regla de Kaiser**: conservar los componentes con autovalor $> 1$ (los que explican
  más que una variable original estandarizada): aquí, 3.
- **Codo del scree plot**: donde la curva se aplana; aquí después de PC3 o PC4.

Lo importante no es la regla sino para **qué** se reduce: para visualizar, 2; para
eliminar redundancia antes de un modelo, los que no pierdan lo que el modelo necesita
(sección 5); para comprimir, los que den un error de reconstrucción aceptable
(sección 4).

## 3. Leer los componentes: cargas y biplot

Cada componente es una combinación lineal de las variables originales; los coeficientes
—las **cargas** (*loadings*)— dicen qué mide. Como todas las variables están
estandarizadas, las cargas son comparables entre sí.

In [ ]:
cargas = pd.DataFrame(pca.components_[:3].T, index=X.columns, columns=["PC1", "PC2", "PC3"])
print(cargas.round(2).to_string())

Z = pca.transform(X_esc)
grupo2 = KMeans(n_clusters=2, n_init=10, random_state=SEMILLA).fit(X_esc).labels_

fig, ejes = plt.subplots(1, 4, figsize=(21, 5))
for eje, color, titulo in [(ejes[0], (tipo == "tinto").astype(int), "Color: tipo (no visto por PCA)"),
                           (ejes[1], grupo2, "Color: grupo K-Means del notebook 02"),
                           (ejes[2], calidad, "Color: quality (3–9)")]:
    disp = eje.scatter(Z[:, 0], Z[:, 1], c=color, cmap="viridis", s=5, alpha=0.6)
    eje.set_xlabel(f"PC1 ({100 * pca.explained_variance_ratio_[0]:.0f} %)")
    eje.set_ylabel(f"PC2 ({100 * pca.explained_variance_ratio_[1]:.0f} %)")
    eje.set_title(titulo)
plt.colorbar(disp, ax=ejes[2], label="quality")
# Cuarto panel: las cargas como flechas (círculo de correlaciones).
eje = ejes[3]
for var, (c1, c2) in cargas[["PC1", "PC2"]].iterrows():
    eje.annotate("", xy=(c1, c2), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="C3", lw=1.5))
    eje.text(c1 * 1.12, c2 * 1.12, var, color="C3", fontsize=8, ha="center", va="center")
eje.add_patch(plt.Circle((0, 0), 1, fill=False, color="gray", ls="--"))
eje.axhline(0, color="gray", lw=0.5)
eje.axvline(0, color="gray", lw=0.5)
eje.set_xlim(-0.75, 0.75)
eje.set_ylim(-0.75, 0.75)
eje.set_aspect("equal")
eje.set_xlabel("carga en PC1")
eje.set_ylabel("carga en PC2")
eje.set_title("Cargas de las 11 variables")
plt.tight_layout()
plt.show()

**PC1** (27 %) carga positivo en los dióxidos de azufre y el azúcar, y negativo en la
acidez volátil, los cloruros y los sulfatos: es el eje **blanco ↔ tinto**, la misma
química que K-Means encontró en el notebook 02. Por eso el panel central es casi idéntico
al de la izquierda. **PC2** (22 %) carga positivo en densidad y azúcar y negativo en
alcohol: el eje **dulce ↔ seco/alcohólico** — el segundo nivel de estructura del notebook
02, otra vez. PCA y K-Means encuentran lo mismo porque ambos buscan lo mismo: la
varianza dominante.

Y el tercer panel muestra lo que ninguno encuentra: la calidad no se ordena a lo largo de
PC1 ni de PC2. Hay un gradiente suave (los vinos de calidad alta tienden a PC2 negativo:
más alcohol), pero nada que separe.

## 4. PCA como compresión: dígitos manuscritos

Con los componentes se puede **reconstruir** cada punto:
$\hat{\mathbf{x}} = \bar{\mathbf{x}} + \mathbf{V}_q \mathbf{V}_q^\top (\mathbf{x} - \bar{\mathbf{x}})$,
con $\mathbf{V}_q$ las primeras $q$ columnas. El error de reconstrucción es exactamente la
varianza de los componentes descartados. Se ve mejor con imágenes: 1797 dígitos de 8 × 8
píxeles (64 dimensiones), `load_digits` de `scikit-learn`.

In [ ]:
digitos = load_digits()
X_dig, y_dig = digitos.data, digitos.target
pca_dig = PCA().fit(X_dig)
acumulada = np.cumsum(pca_dig.explained_variance_ratio_)
print("Varianza acumulada con q componentes:",
      {q: round(float(acumulada[q - 1]), 3) for q in [2, 5, 10, 20, 30, 40, 64]})


def reconstruir(pca_ajustado, X, q):
    Vq = pca_ajustado.components_[:q]
    return pca_ajustado.mean_ + (X - pca_ajustado.mean_) @ Vq.T @ Vq


qs = [1, 2, 5, 10, 20, 40, 64]
indices = [0, 1, 2, 3, 4, 5]              # un ejemplar de los dígitos 0–5
fig, ejes = plt.subplots(len(indices), len(qs) + 1, figsize=(1.6 * (len(qs) + 1), 1.6 * len(indices)))
for fila, i in enumerate(indices):
    ejes[fila, 0].imshow(X_dig[i].reshape(8, 8), cmap="gray_r")
    ejes[fila, 0].set_title("original" if fila == 0 else "")
    for col, q in enumerate(qs, start=1):
        ejes[fila, col].imshow(reconstruir(pca_dig, X_dig[i:i + 1], q).reshape(8, 8), cmap="gray_r")
        if fila == 0:
            ejes[fila, col].set_title(f"q = {q}")
for eje in ejes.ravel():
    eje.set_xticks([])
    eje.set_yticks([])
plt.show()

error = [np.mean((X_dig - reconstruir(pca_dig, X_dig, q)) ** 2) for q in range(1, 65)]
varianza_descartada = [pca_dig.explained_variance_[q:].sum() * (len(X_dig) - 1) / len(X_dig) / 64 for q in range(1, 65)]
print("El error cuadrático medio de reconstrucción es la varianza descartada:",
      np.allclose(error, varianza_descartada))
fig, eje = plt.subplots(figsize=(7, 4))
eje.plot(range(1, 65), error, "o-", ms=3)
eje.set_xlabel("Componentes conservados q")
eje.set_ylabel("Error cuadrático medio por píxel")
eje.set_title("Error de reconstrucción: cae rápido y luego lentamente")
plt.show()

Con 10 componentes de 64 (el 74 % de la varianza) los dígitos ya se reconocen; con 20
(89 %) son casi indistinguibles del original. El error cae rápido porque los píxeles
vecinos están muy correlacionados —los trazos son continuos— y eso es exactamente lo
que PCA explota. Es la razón por la que PCA sirve para comprimir imágenes, y también
para **eliminar ruido**: los últimos componentes son variaciones píxel a píxel que no
forman trazos.

## 5. Dos límites de PCA, medidos

### 5.1 Variables incorreladas: no hay nada que comprimir

PCA comprime porque hay correlación: si dos variables se mueven juntas, una dirección
captura a ambas. Sin correlación, cada componente vale tanto como una variable. Los seis
predictores de `rendimiento-estudiantes.csv` (módulos 1 y 3) fueron generados
**independientes** (el módulo 1 lo señaló en el ejercicio 02):

In [ ]:
estudiantes = pd.read_csv("../datos/rendimiento-estudiantes.csv")
predictores = ["edad", "estrato", "trabaja", "promedio_anterior", "horas_estudio_semana", "asistencia_pct"]
pca_est = PCA().fit(StandardScaler().fit_transform(estudiantes[predictores]))
print("Correlación máxima entre predictores (en valor absoluto):",
      round(float(estudiantes[predictores].corr().abs().where(~np.eye(6, dtype=bool)).max().max()), 3))
print("Varianza explicada por componente:", (100 * pca_est.explained_variance_ratio_).round(1), "%")
print(f"Con 2 componentes: {100 * pca_est.explained_variance_ratio_[:2].sum():.0f} % · "
      f"Wine Quality con 2: {100 * pca.explained_variance_ratio_[:2].sum():.0f} %")

El scree plot es casi plano: 24, 18, 17, 15, 14, 11 %. Dos componentes retienen el 43 %
(frente al 50 % de Wine, cuyas variables sí están correlacionadas) y cinco de seis hacen
falta para llegar al 90 %. Aplicar PCA aquí no reduce nada; solo cambia los ejes por
combinaciones de variables que ya no se pueden interpretar. Antes de usar PCA, mirar la
matriz de correlaciones (módulo 2): si está vacía, PCA no tiene qué encontrar.

### 5.2 La información puede estar en la dirección de menor varianza

El límite más importante. PCA **no ve $y$**. Si lo que separa las clases es una dirección
de poca varianza, PCA la descarta primero. Un ejemplo construido: dos clases, separadas
solo en la segunda variable, que tiene 36 veces menos varianza que la primera.

In [ ]:
n = 400
y_toy = np.repeat([0, 1], n // 2)
X_toy = np.c_[rng.normal(0, 3, n), rng.normal(0, 0.5, n) + 1.6 * y_toy]
pca_toy = PCA().fit(X_toy)
Z_toy = pca_toy.transform(X_toy)
print("Varianza explicada:", pca_toy.explained_variance_ratio_.round(3))
for j in range(2):
    acc = cross_val_score(LogisticRegression(), Z_toy[:, [j]], y_toy, cv=5).mean()
    print(f"Regresión logística usando solo PC{j + 1}: accuracy = {acc:.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap="coolwarm", s=8)
ejes[0].set_title("Dos clases separadas en la variable de poca varianza")
ejes[0].set_xlabel("x₁ (varianza 9)")
ejes[0].set_ylabel("x₂ (varianza 0.25 + separación)")
ejes[1].scatter(Z_toy[:, 0], rng.normal(0, 0.02, n), c=y_toy, cmap="coolwarm", s=8)
ejes[1].set_title(f"Proyección sobre PC1 ({100 * pca_toy.explained_variance_ratio_[0]:.0f} % de la varianza): las clases se mezclan")
ejes[1].set_xlabel("PC1")
ejes[1].set_yticks([])
plt.show()

PC1 explica el 91 % de la varianza y no sirve para nada (accuracy 0.47, el azar); PC2,
con el 9 %, separa las clases (0.93). "Reducir a los componentes que explican el 90 %"
habría tirado exactamente la información útil. Este es el argumento contra usar PCA a
ciegas como preprocesamiento de un modelo supervisado.

### 5.3 …y con la calidad del vino pasa lo mismo, en pequeño

Regresión logística para `quality ≥ 7` (el problema del módulo 4), con PCA dentro del
`Pipeline` —ajustado solo sobre entrenamiento, como cualquier transformación— y distintos
números de componentes:

In [ ]:
y_buena = (calidad >= 7).astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
filas = []
for q in [2, 3, 5, 7, 9, 11]:
    modelo = make_pipeline(StandardScaler(), PCA(n_components=q), LogisticRegression(max_iter=2000))
    filas.append({"componentes": q, "% varianza": 100 * np.cumsum(pca.explained_variance_ratio_)[q - 1],
                  "AP": cross_val_score(modelo, X, y_buena, cv=cv, scoring="average_precision").mean()})
sin_pca = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
filas.append({"componentes": "sin PCA", "% varianza": 100.0,
              "AP": cross_val_score(sin_pca, X, y_buena, cv=cv, scoring="average_precision").mean()})
print(pd.DataFrame(filas).round(3).to_string(index=False))

Con los 11 componentes la AP es **idéntica** a la del modelo sin PCA: PCA con todos los
componentes es una rotación, y un modelo lineal es invariante a rotaciones. Con menos
componentes, la AP baja: de 0.52 a 0.50 con 7 (el 90 % de la varianza), a 0.46 con 5, a
0.40 con 2. Los componentes de poca varianza —que las reglas de la sección 2 descartan—
contienen parte de lo que predice la calidad. Aquí la pérdida es moderada; en el ejemplo
sintético era total. La regla práctica: PCA antes de un modelo supervisado es una
decisión que **se valida** con la métrica del modelo, no con la varianza explicada.

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Qué es un componente? | Un autovector de la covarianza (o una columna de $\mathbf{V}$ en la SVD); el autovalor es su varianza. A mano y `scikit-learn` coinciden a $10^{-10}$, salvo signo |
| ¿Estandarizar? | Sin estandarizar, PC1 de Wine es `total_sulfur_dioxide` con el 95 %; estandarizado, PC1 es el eje tinto/blanco con el 27 % |
| ¿Cuántos componentes? | 80 % de varianza: 5; Kaiser: 3; codo: 3–4. La regla correcta depende de para qué |
| ¿Qué miden PC1 y PC2 de Wine? | Tinto ↔ blanco y dulce ↔ seco: lo mismo que K-Means, porque ambos buscan la varianza dominante. La calidad no aparece |
| ¿Compresión? | Dígitos 8 × 8: 10 de 64 componentes bastan para reconocerlos; el error de reconstrucción es la varianza descartada |
| ¿Cuándo no sirve? | Variables incorreladas (estudiantes: scree plano, 43 % con 2 componentes) |
| ¿Cuándo daña? | Cuando $y$ vive en una dirección de poca varianza: PC1 (91 %) da accuracy 0.47, PC2 (9 %) da 0.93. En Wine, quitar componentes baja la AP de 0.52 a 0.40 |